# Phase 1: Data Preparation — Kaggle Version
## Voice Biometric Authentication System

### Before running — add these datasets in Kaggle:
1. Click **+ Add Data** (top right)
2. Search **`asvpoof-2019-dataset`** → Add
3. Search **`librispeech-clean`** → Add
4. Upload your own voice zip → Add

### Then enable Internet:
- Right panel → Session options → Internet ON

### Enable GPU:
- Right panel → Accelerator → GPU P100

In [ ]:
# ── Debug: Print everything Kaggle sees ───────────────────────────────────────
import os, glob

print('=== /kaggle/input/ contents ===')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:                          # only show files 3 levels deep
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... ({len(files)-5} more files)')


In [ ]:
!pip install -q librosa soundfile webrtcvad
print('Done.')

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import json
import pickle
import struct
import webrtcvad
from pathlib import Path
from sklearn.model_selection import train_test_split
from IPython.display import Audio, display

# ── Kaggle Paths ──────────────────────────────────────────────────────────────
WORKING_DIR  = '/kaggle/working'
INPUT_DIR    = '/kaggle/input'

# Output folders (persist as notebook output)
BASE_DIR = f'{WORKING_DIR}/voice_auth'

dirs = [
    f'{BASE_DIR}/data/processed/genuine',
    f'{BASE_DIR}/data/processed/impostor',
    f'{BASE_DIR}/data/processed/spoof',
    f'{BASE_DIR}/data/features/mfcc',
    f'{BASE_DIR}/data/features/melspec',
    f'{BASE_DIR}/models',
    f'{BASE_DIR}/results',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# ── Audio Config ──────────────────────────────────────────────────────────────
SAMPLE_RATE = 16000
N_MFCC      = 40
N_MELS      = 80
HOP_LENGTH  = 160
WIN_LENGTH  = 400
N_FFT       = 512
CLIP_DUR    = 3.0

SPEAKER_NAME = 'abhiram'   # change to your name

print(f'Working dir : {BASE_DIR}')
print(f'Sample rate : {SAMPLE_RATE} Hz')

## Step 1: Find Your Voice Data
Upload your `my_voice_data.zip` as a Kaggle dataset, then run this cell.

In [ ]:
import zipfile, shutil

VOICE_RAW_DIR = f'{BASE_DIR}/data/raw/genuine'
os.makedirs(VOICE_RAW_DIR, exist_ok=True)

# ── Step 1: Try to find a zip and extract it ──────────────────────────────────
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zips:
    print(f'Found zip: {zips[0]}')
    with zipfile.ZipFile(zips[0], 'r') as z:
        z.extractall(VOICE_RAW_DIR)
    print('Extracted.')

# ── Step 2: Kaggle auto-extracts zips — search WAV files directly ─────────────
# Find all WAV files anywhere under /kaggle/input/
all_input_wavs = glob.glob('/kaggle/input/**/*.wav', recursive=True)
print(f'\nWAV files found in /kaggle/input/: {len(all_input_wavs)}')
for f in all_input_wavs[:10]:
    print(f'  {f}')

# Copy them to our working directory
if all_input_wavs:
    copied = 0
    for src in all_input_wavs:
        dst = f'{VOICE_RAW_DIR}/{Path(src).name}'
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
    print(f'\nCopied {copied} WAV files to {VOICE_RAW_DIR}')

# ── Step 3: Final count ───────────────────────────────────────────────────────
voice_files = glob.glob(f'{VOICE_RAW_DIR}/**/*.wav', recursive=True)
print(f'Your voice files ready: {len(voice_files)}')

if len(voice_files) == 0:
    print()
    print('STILL 0? Check these steps:')
    print('  1. Click + Add Data in Kaggle')
    print('  2. Go to Your Datasets tab (not Search)')
    print('  3. Select your uploaded voice dataset')
    print('  4. Restart kernel and run again')


## Step 2: Load LibriSpeech (Impostor Voices)

In [ ]:
from collections import defaultdict

# Find LibriSpeech in attached datasets
libri_paths = glob.glob(f'{INPUT_DIR}/*libri*/**/*.flac', recursive=True) + \
              glob.glob(f'{INPUT_DIR}/*libri*/**/*.wav',  recursive=True)

if libri_paths:
    print(f'LibriSpeech files found: {len(libri_paths)}')
else:
    # Download directly (Internet must be ON)
    print('LibriSpeech not found in datasets. Downloading test-clean (~346MB)...')
    import torchaudio
    libri_dl_dir = f'{BASE_DIR}/data/raw/librispeech'
    os.makedirs(libri_dl_dir, exist_ok=True)
    libri_dataset = torchaudio.datasets.LIBRISPEECH(
        root=libri_dl_dir, url='test-clean', download=True
    )
    libri_paths = glob.glob(f'{libri_dl_dir}/**/*.flac', recursive=True)
    print(f'Downloaded: {len(libri_paths)} files')

# Group by speaker (folder name = speaker id in LibriSpeech)
speaker_files = defaultdict(list)
for f in libri_paths:
    spk = Path(f).parts[-3]  # LibriSpeech folder structure: speaker/chapter/file
    speaker_files[spk].append(f)

print(f'Unique speakers : {len(speaker_files)}')

In [ ]:
# Save up to 40 speakers × 50 utterances = ~2000 impostor WAVs
# LibriSpeech test-clean has 40 speakers, avg 65 utterances each
IMPOSTOR_DIR = f'{BASE_DIR}/data/processed/impostor'
NUM_SPK = 40   # all speakers
NUM_UTT = 50   # utterances per speaker (max)

selected_spks = list(speaker_files.keys())[:NUM_SPK]
saved = 0
for spk in selected_spks:
    utts = speaker_files[spk][:NUM_UTT]
    for i, f in enumerate(utts):
        out = f'{IMPOSTOR_DIR}/spk{spk}_{i:03d}.wav'
        if os.path.exists(out):
            saved += 1
            continue
        try:
            audio, sr = librosa.load(f, sr=SAMPLE_RATE, mono=True)
            audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.9
            target = int(SAMPLE_RATE * CLIP_DUR)
            audio  = audio[:target] if len(audio) >= target else np.pad(audio, (0, target - len(audio)))
            sf.write(out, audio, SAMPLE_RATE)
            saved += 1
        except Exception as e:
            print(f'  Error {f}: {e}')

print(f'Impostor files saved: {saved}')
print(f'Unique speakers used: {len(selected_spks)}')

## Step 3: Load ASVspoof 2019 (Spoof / DeepFake)

In [ ]:
SPOOF_OUT = f'{BASE_DIR}/data/processed/spoof'

# Try attached Kaggle dataset first
asv_protocols = glob.glob(f'{INPUT_DIR}/*asvspoof*/**/*.txt', recursive=True) + \
                glob.glob(f'{INPUT_DIR}/*asvpoof*/**/*.txt',  recursive=True)
asv_audio     = glob.glob(f'{INPUT_DIR}/*asvspoof*/**/*.flac', recursive=True) + \
                glob.glob(f'{INPUT_DIR}/*asvpoof*/**/*.flac',  recursive=True)

print(f'ASVspoof protocol files : {len(asv_protocols)}')
print(f'ASVspoof audio files    : {len(asv_audio)}')

if asv_audio:
    audio_lookup = {Path(f).stem: f for f in asv_audio}

    if asv_protocols:
        df_proto = pd.read_csv(asv_protocols[0], sep=' ', header=None,
                               names=['speaker','file','env','attack','label'])
        print('Label counts:', df_proto['label'].value_counts().to_dict())
        spoof_ids   = df_proto[df_proto['label']=='spoof'  ]['file'].tolist()[:500]
        genuine_ids = df_proto[df_proto['label']=='bonafide']['file'].tolist()[:200]
    else:
        spoof_ids   = list(audio_lookup.keys())[:500]
        genuine_ids = []

    saved_s = saved_g = 0
    for fid in spoof_ids:
        src = audio_lookup.get(fid)
        if src is None: continue
        audio, sr = librosa.load(src, sr=SAMPLE_RATE, mono=True)
        audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.9
        target = int(SAMPLE_RATE * CLIP_DUR)
        audio  = audio[:target] if len(audio) >= target else np.pad(audio, (0, target-len(audio)))
        sf.write(f'{SPOOF_OUT}/asv_spoof_{fid}.wav', audio, SAMPLE_RATE)
        saved_s += 1

    print(f'ASVspoof files saved: {saved_s} spoof')

else:
    print('ASVspoof not found. Generating TTS spoof instead...')

In [ ]:
# Fallback: Generate spoof with gTTS if ASVspoof not attached
spoof_existing = glob.glob(f'{SPOOF_OUT}/*.wav')

if len(spoof_existing) < 50:
    print('Generating TTS spoof samples with gTTS...')
    !pip install -q gtts pydub
    from gtts import gTTS
    from pydub import AudioSegment
    import io

    PHRASES = [
        'My voice is my password',
        'Open sesame authenticate now',
        'Verify my identity please',
        'The quick brown fox jumps over the lazy dog',
        'Speech recognition systems use deep neural networks',
        'Biometric authentication verifies human identity',
        'Count from one to twenty slowly',
        'Deep learning has revolutionised speaker verification',
        'Hello my name is a computer generated voice',
        'This is a synthetic speech sample for testing',
    ]
    TLDS = ['com', 'co.uk', 'com.au', 'co.in', 'ca']

    count = 0
    for t_idx, tld in enumerate(TLDS):
        for p_idx, phrase in enumerate(PHRASES):
            out_path = f'{SPOOF_OUT}/gtts_{tld.replace(".","_")}_{p_idx:02d}.wav'
            if os.path.exists(out_path):
                count += 1
                continue
            try:
                tts_g  = gTTS(text=phrase, lang='en', tld=tld)
                buf    = io.BytesIO()
                tts_g.write_to_fp(buf)
                buf.seek(0)
                seg = AudioSegment.from_mp3(buf)
                wav_buf = io.BytesIO()
                seg.export(wav_buf, format='wav')
                wav_buf.seek(0)
                audio, sr = librosa.load(wav_buf, sr=SAMPLE_RATE, mono=True)
                audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.9
                target = int(SAMPLE_RATE * CLIP_DUR)
                audio  = audio[:target] if len(audio) >= target else np.pad(audio,(0,target-len(audio)))
                sf.write(out_path, audio, SAMPLE_RATE)
                count += 1
            except Exception as e:
                print(f'  gTTS error ({tld}/{p_idx}): {e}')

    print(f'Generated {count} TTS spoof samples.')
else:
    print(f'Already have {len(spoof_existing)} spoof files. Skipping TTS generation.')

## Step 4: Preprocess Your Voice (Genuine)

In [ ]:
# Step 4: Preprocess Genuine Voice — Keep text-dependent / text-independent split
def preprocess_audio(filepath, target_sr=16000, duration=3.0):
    try:
        audio, sr = librosa.load(filepath, sr=target_sr, mono=True)
        audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.9
        audio, _ = librosa.effects.trim(audio, top_db=20)
        target_len = int(target_sr * duration)
        audio = audio[:target_len] if len(audio) >= target_len \
                else np.pad(audio, (0, target_len - len(audio)))
        return audio.astype(np.float32)
    except Exception as e:
        print(f'  Error: {filepath} — {e}')
        return None


GENUINE_OUT = f'{BASE_DIR}/data/processed/genuine'
voice_raw   = glob.glob(f'{VOICE_RAW_DIR}/**/*.wav', recursive=True)
print(f'Processing {len(voice_raw)} genuine files...')

td_count = 0   # text-dependent (phrase recordings)
ti_count = 0   # text-independent (free speech)

for f in voice_raw:
    audio = preprocess_audio(f)
    if audio is None:
        continue

    # Detect type from path: phrase1/phrase2/phrase3 = text-dependent
    path_str = str(f).lower()
    if any(p in path_str for p in ['phrase1', 'phrase2', 'phrase3']):
        prefix = 'td_'   # text-dependent
        td_count += 1
    else:
        prefix = 'ti_'   # text-independent (free speech)
        ti_count += 1

    out = f'{GENUINE_OUT}/{prefix}{Path(f).stem}.wav'
    sf.write(out, audio, SAMPLE_RATE)

print(f'Text-dependent  (td_*): {td_count}   ← used by Phase 2')
print(f'Text-independent (ti_*): {ti_count}  ← used by Phase 3')
print(f'Total genuine processed: {td_count + ti_count}')

## Step 5: Feature Extraction (MFCC + Mel-Spectrogram)

In [ ]:
def extract_mfcc(audio, sr=16000):
    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC,
                                   n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return np.vstack([mfcc, delta, delta2])  # (120, T)

def extract_melspec(audio, sr=16000):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    return librosa.power_to_db(mel, ref=np.max)  # (80, T)

all_records = []

label_dirs = {
    'genuine' : f'{BASE_DIR}/data/processed/genuine',
    'impostor': f'{BASE_DIR}/data/processed/impostor',
    'spoof'   : f'{BASE_DIR}/data/processed/spoof',
}

for label, src_dir in label_dirs.items():
    files = glob.glob(f'{src_dir}/*.wav')
    print(f'Extracting features for {label}: {len(files)} files')
    for f in files:
        audio, _ = librosa.load(f, sr=SAMPLE_RATE)
        mfcc_feat = extract_mfcc(audio)
        mel_feat  = extract_melspec(audio)
        base      = Path(f).stem
        mfcc_path = f'{BASE_DIR}/data/features/mfcc/{base}.npy'
        mel_path  = f'{BASE_DIR}/data/features/melspec/{base}.npy'
        np.save(mfcc_path, mfcc_feat)
        np.save(mel_path,  mel_feat)
        all_records.append({
            'audio_file'  : f,
            'mfcc_file'   : mfcc_path,
            'melspec_file': mel_path,
            'label'       : label,
        })

df_all = pd.DataFrame(all_records)
df_all.to_csv(f'{BASE_DIR}/data/features_index.csv', index=False)
print(f'\nTotal features extracted: {len(df_all)}')
print(df_all['label'].value_counts())

In [ ]:
# ── Dataset Balancing — Target 2000 per class (6000 total) ───────────────────
import random, itertools

TARGET = 2000  # samples per class

# 14 augmentation types for rich variety
AUG_TYPES = [
    'noise_low',       # SNR ~25 dB  — barely audible
    'noise_mid',       # SNR ~15 dB  — light noise
    'noise_high',      # SNR ~8 dB   — noisy
    'pitch_up1',       # +1 semitone
    'pitch_up2',       # +2 semitones
    'pitch_down1',     # -1 semitone
    'pitch_down2',     # -2 semitones
    'speed_fast',      # 1.15× speed
    'speed_slow',      # 0.85× speed
    'noise_pitch_up',  # noise + pitch +1
    'noise_pitch_dn',  # noise + pitch -1
    'noise_speed_up',  # noise + speed 1.1×
    'noise_speed_dn',  # noise + speed 0.9×
    'reverb',          # simple room reverb
]

def augment_audio(audio, sr, aug_type):
    audio = audio.astype(np.float32)
    try:
        if aug_type == 'noise_low':
            return audio + np.random.normal(0, 0.002, len(audio)).astype(np.float32)
        elif aug_type == 'noise_mid':
            return audio + np.random.normal(0, 0.007, len(audio)).astype(np.float32)
        elif aug_type == 'noise_high':
            return audio + np.random.normal(0, 0.015, len(audio)).astype(np.float32)
        elif aug_type == 'pitch_up1':
            return librosa.effects.pitch_shift(audio, sr=sr, n_steps=1).astype(np.float32)
        elif aug_type == 'pitch_up2':
            return librosa.effects.pitch_shift(audio, sr=sr, n_steps=2).astype(np.float32)
        elif aug_type == 'pitch_down1':
            return librosa.effects.pitch_shift(audio, sr=sr, n_steps=-1).astype(np.float32)
        elif aug_type == 'pitch_down2':
            return librosa.effects.pitch_shift(audio, sr=sr, n_steps=-2).astype(np.float32)
        elif aug_type == 'speed_fast':
            s = librosa.effects.time_stretch(audio, rate=1.15)
            return librosa.util.fix_length(s, size=len(audio)).astype(np.float32)
        elif aug_type == 'speed_slow':
            s = librosa.effects.time_stretch(audio, rate=0.85)
            return librosa.util.fix_length(s, size=len(audio)).astype(np.float32)
        elif aug_type == 'noise_pitch_up':
            a = audio + np.random.normal(0, 0.004, len(audio)).astype(np.float32)
            return librosa.effects.pitch_shift(a, sr=sr, n_steps=1).astype(np.float32)
        elif aug_type == 'noise_pitch_dn':
            a = audio + np.random.normal(0, 0.004, len(audio)).astype(np.float32)
            return librosa.effects.pitch_shift(a, sr=sr, n_steps=-1).astype(np.float32)
        elif aug_type == 'noise_speed_up':
            a = audio + np.random.normal(0, 0.003, len(audio)).astype(np.float32)
            s = librosa.effects.time_stretch(a, rate=1.1)
            return librosa.util.fix_length(s, size=len(audio)).astype(np.float32)
        elif aug_type == 'noise_speed_dn':
            a = audio + np.random.normal(0, 0.003, len(audio)).astype(np.float32)
            s = librosa.effects.time_stretch(a, rate=0.9)
            return librosa.util.fix_length(s, size=len(audio)).astype(np.float32)
        elif aug_type == 'reverb':
            ir_len  = int(sr * 0.25)
            ir      = np.exp(-5 * np.linspace(0, 1, ir_len)) * np.random.randn(ir_len)
            out_sig = np.convolve(audio, ir.astype(np.float32), mode='full')[:len(audio)]
            return (out_sig / (np.max(np.abs(out_sig)) + 1e-8) * 0.9).astype(np.float32)
    except Exception as e:
        pass
    return audio

def fill_to_target(src_wavs, out_dir, prefix, target, aug_types, sr=16000):
    existing = glob.glob(f'{out_dir}/{prefix}*.wav')
    needed   = target - len(glob.glob(f'{out_dir}/*.wav'))
    if needed <= 0:
        print(f'  Already at target ({len(glob.glob(f"{out_dir}/*.wav"))} files)')
        return 0
    print(f'  Need {needed} more augmented files...')
    aug_cycle = itertools.cycle(aug_types)
    saved = 0
    for i in range(needed):
        src      = src_wavs[i % len(src_wavs)]
        aug_type = next(aug_cycle)
        try:
            audio, _ = librosa.load(src, sr=sr, mono=True)
            aug      = augment_audio(audio, sr, aug_type)
            aug      = aug / (np.max(np.abs(aug)) + 1e-8) * 0.9
            tag      = aug_type.replace('+', '_')
            out_path = f'{out_dir}/{prefix}{i:05d}_{tag}.wav'
            if not os.path.exists(out_path):
                sf.write(out_path, aug, sr)
            saved += 1
        except Exception as e:
            pass
        if (i + 1) % 200 == 0:
            print(f'    {i+1}/{needed} done...')
    return saved


# ─── Part A: Augment GENUINE to 2000 ─────────────────────────────────────────
print('=== Part A: Genuine augmentation ===')
genuine_orig = [f for f in glob.glob(f'{BASE_DIR}/data/processed/genuine/*.wav')
                if 'aug_' not in Path(f).stem]
print(f'Original genuine files : {len(genuine_orig)}')
n_saved = fill_to_target(genuine_orig,
                          f'{BASE_DIR}/data/processed/genuine',
                          'aug_', TARGET, AUG_TYPES)
total_genuine = len(glob.glob(f'{BASE_DIR}/data/processed/genuine/*.wav'))
print(f'Augmented saved        : {n_saved}')
print(f'Total genuine now      : {total_genuine}')


# ─── Part B: Generate more SPOOF gTTS then augment to 2000 ───────────────────
print('\n=== Part B: Spoof generation ===')
try:
    from gtts import gTTS
    from pydub import AudioSegment
    import io

    SPOOF_PHRASES = [
        'My voice is my password verify me',
        'Open sesame authenticate now please',
        'Verify my identity with this phrase',
        'The quick brown fox jumps over the lazy dog',
        'Speech recognition systems use deep learning models',
        'Biometric authentication verifies human identity securely',
        'Count from one to twenty slowly and clearly',
        'Deep learning has revolutionised speaker verification systems',
        'Hello this is a computer generated voice recording',
        'This synthetic speech sample is used for spoof detection testing',
        'Speaker verification requires accurate acoustic modelling techniques',
        'Voice activity detection removes silence segments from audio recordings',
        'Mel frequency cepstral coefficients are widely used speech features',
        'Neural networks can distinguish between real and synthesized voices',
        'Authentication systems must be robust against spoofing attacks',
        'My passphrase for today is both secure and unique',
        'Please verify that this voice belongs to the correct person',
        'The anti-spoofing system successfully rejected the fake voice sample',
        'Cosine similarity is used to measure speaker embedding distances',
        'Equal error rate is the standard metric for biometric evaluation',
        'Voice biometrics can be used for secure access control systems',
        'Automatic speaker verification is an important security technology',
        'The neural network model was trained on thousands of hours of speech',
        'I am speaking clearly so that the system can recognize my voice',
        'Security systems protect sensitive information from unauthorized access',
        'Speech synthesis has improved dramatically with modern deep learning',
        'The microphone captures audio signals for processing by the system',
        'Feature extraction transforms raw audio into numerical representations',
        'Classification algorithms assign labels to extracted audio features',
        'Training data quality is crucial for building accurate recognition models',
    ]
    TLDS = ['com', 'co.uk', 'com.au', 'co.in', 'ca', 'ie', 'co.nz', 'co.za', 'co.ke', 'us']

    spoof_dir   = f'{BASE_DIR}/data/processed/spoof'
    count_gtts  = 0
    for tld in TLDS:
        for p_idx, phrase in enumerate(SPOOF_PHRASES):
            tag      = tld.replace('.', '_')
            out_path = f'{spoof_dir}/gtts_{tag}_{p_idx:03d}.wav'
            if os.path.exists(out_path):
                count_gtts += 1
                continue
            try:
                tts_g  = gTTS(text=phrase, lang='en', tld=tld)
                buf    = io.BytesIO()
                tts_g.write_to_fp(buf); buf.seek(0)
                seg     = AudioSegment.from_mp3(buf)
                wav_buf = io.BytesIO()
                seg.export(wav_buf, format='wav'); wav_buf.seek(0)
                audio, _  = librosa.load(wav_buf, sr=SAMPLE_RATE, mono=True)
                audio     = audio / (np.max(np.abs(audio)) + 1e-8) * 0.9
                t_len     = int(SAMPLE_RATE * CLIP_DUR)
                audio     = audio[:t_len] if len(audio) >= t_len else np.pad(audio, (0, t_len - len(audio)))
                sf.write(out_path, audio, SAMPLE_RATE)
                count_gtts += 1
            except Exception as e:
                pass  # skip on gTTS rate-limit or error
    print(f'gTTS files ready: {count_gtts}')
except ImportError:
    print('gTTS not installed — skipping gTTS generation')

spoof_base = [f for f in glob.glob(f'{BASE_DIR}/data/processed/spoof/*.wav')
              if 'spoof_aug_' not in Path(f).stem]
print(f'Base spoof files: {len(spoof_base)}')
n_saved = fill_to_target(spoof_base,
                          f'{BASE_DIR}/data/processed/spoof',
                          'spoof_aug_', TARGET, AUG_TYPES)
total_spoof = len(glob.glob(f'{BASE_DIR}/data/processed/spoof/*.wav'))
print(f'Total spoof now : {total_spoof}')


# ─── Part C: Augment IMPOSTOR to 2000 if needed ──────────────────────────────
print('\n=== Part C: Impostor augmentation (if < 2000) ===')
impostor_orig = glob.glob(f'{BASE_DIR}/data/processed/impostor/*.wav')
print(f'Original impostor files: {len(impostor_orig)}')
# Use only fast augmentations for impostors (noise + speed — no pitch shift needed)
FAST_AUGS = ['noise_low', 'noise_mid', 'noise_high', 'speed_fast',
             'speed_slow', 'noise_speed_up', 'noise_speed_dn']
n_saved = fill_to_target(impostor_orig,
                          f'{BASE_DIR}/data/processed/impostor',
                          'imp_aug_', TARGET, FAST_AUGS)
total_impostor = len(glob.glob(f'{BASE_DIR}/data/processed/impostor/*.wav'))
print(f'Total impostor now : {total_impostor}')


# ─── Part D: Re-extract features for new files only ──────────────────────────
print('\n=== Part D: Feature extraction for new files ===')
existing_csv = pd.read_csv(f'{BASE_DIR}/data/features_index.csv')
already_done = set(existing_csv['audio_file'].tolist())
new_records  = []

for label, src_dir in label_dirs.items():
    new_files = [f for f in glob.glob(f'{src_dir}/*.wav') if f not in already_done]
    print(f'  {label}: {len(new_files)} new files to process')
    for f in new_files:
        try:
            audio, _ = librosa.load(f, sr=SAMPLE_RATE)
            base      = Path(f).stem
            mfcc_feat = extract_mfcc(audio)
            mel_feat  = extract_melspec(audio)
            mfcc_path = f'{BASE_DIR}/data/features/mfcc/{base}.npy'
            mel_path  = f'{BASE_DIR}/data/features/melspec/{base}.npy'
            np.save(mfcc_path, mfcc_feat)
            np.save(mel_path,  mel_feat)
            new_records.append({'audio_file': f, 'mfcc_file': mfcc_path,
                                 'melspec_file': mel_path, 'label': label})
        except Exception as e:
            pass

if new_records:
    df_new      = pd.DataFrame(new_records)
    df_combined = pd.concat([existing_csv, df_new], ignore_index=True)
    df_combined.to_csv(f'{BASE_DIR}/data/features_index.csv', index=False)
    print(f'\nAdded {len(new_records)} new feature files.')
else:
    df_combined = existing_csv
    print('No new files to process.')

print('\n=== Final Dataset ===')
counts = df_combined['label'].value_counts()
print(counts)
print(f'\nTotal: {len(df_combined)} samples')
for lbl, cnt in counts.items():
    bar = '█' * (cnt // 50) + f' {cnt}'
    print(f'  {lbl:10s}: {bar}')

In [ ]:
# ── Dataset Size Check ────────────────────────────────────────────────────────
import os, glob
from pathlib import Path

BASE_DIR     = '/kaggle/working/voice_auth'
GENUINE_DIR  = f'{BASE_DIR}/data/processed/genuine'
IMPOSTOR_DIR = f'{BASE_DIR}/data/processed/impostor'
SPOOF_DIR    = f'{BASE_DIR}/data/processed/spoof'

def count_wavs(directory):
    wavs = glob.glob(f'{directory}/*.wav')
    orig = [f for f in wavs if not any(p in Path(f).stem for p in ['aug_', 'imp_aug_', 'spoof_aug_'])]
    aug  = len(wavs) - len(orig)
    return len(wavs), len(orig), aug

g_total, g_orig, g_aug = count_wavs(GENUINE_DIR)
i_total, i_orig, i_aug = count_wavs(IMPOSTOR_DIR)
s_total, s_orig, s_aug = count_wavs(SPOOF_DIR)
grand_total = g_total + i_total + s_total

print('=' * 55)
print('  DATASET SIZE SUMMARY')
print('=' * 55)
print(f'  {"Class":<12} {"Original":>10} {"Augmented":>11} {"Total":>8}')
print(f'  {"-"*12} {"-"*10} {"-"*11} {"-"*8}')
print(f'  {"Genuine":<12} {g_orig:>10} {g_aug:>11} {g_total:>8}')
print(f'  {"Impostor":<12} {i_orig:>10} {i_aug:>11} {i_total:>8}')
print(f'  {"Spoof":<12} {s_orig:>10} {s_aug:>11} {s_total:>8}')
print(f'  {"-"*12} {"-"*10} {"-"*11} {"-"*8}')
print(f'  {"TOTAL":<12} {"":>10} {"":>11} {grand_total:>8}')
print('=' * 55)

# Duration estimate (assumes 3s clips)
total_mins = grand_total * 3 / 60
print(f'\n  Audio duration : ~{total_mins:.0f} minutes ({grand_total} × 3s clips)')
print(f'  Target per class: 2000  |  Balance: {"OK" if min(g_total,i_total,s_total) >= 1800 else "NOT YET"}')

# Show augmentation breakdown for genuine
print(f'\n  Genuine augmentation types found:')
aug_tags = {}
for f in glob.glob(f'{GENUINE_DIR}/aug_*.wav'):
    tag = '_'.join(Path(f).stem.split('_')[2:])  # extract aug type from filename
    aug_tags[tag] = aug_tags.get(tag, 0) + 1
for tag, cnt in sorted(aug_tags.items()):
    print(f'    {tag:<20}: {cnt}')

# Check features index
feat_csv = f'{BASE_DIR}/data/features_index.csv'
if os.path.exists(feat_csv):
    import pandas as pd
    df = pd.read_csv(feat_csv)
    print(f'\n  Features index: {len(df)} rows')
    print(df['label'].value_counts().to_string())
else:
    print(f'\n  features_index.csv not found yet — run cell 15 first')

## Step 6: Visualize Features

In [ ]:
# Dataset distribution
counts = df_all['label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c','#3498db'], startangle=90)
axes[0].set_title('Class Distribution')

bars = axes[1].bar(counts.index, counts.values,
                   color=['#2ecc71','#e74c3c','#3498db'], edgecolor='black')
for bar, v in zip(bars, counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 str(v), ha='center', fontweight='bold')
axes[1].set_ylabel('Samples')
axes[1].set_title('Sample Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/results/dataset_distribution.png', dpi=150)
plt.show()

In [ ]:
# Compare MFCC across classes
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('MFCC & Mel-Spectrogram per Class', fontsize=14, fontweight='bold')
label_colors = {'genuine': 'Greens', 'impostor': 'Blues', 'spoof': 'Reds'}

for row, label in enumerate(['genuine', 'impostor', 'spoof']):
    subset = df_all[df_all['label'] == label]
    if len(subset) == 0:
        continue
    row_data = subset.iloc[0]
    mfcc_d   = np.load(row_data['mfcc_file'])[:40]
    mel_d    = np.load(row_data['melspec_file'])

    librosa.display.specshow(mfcc_d, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', ax=axes[row, 0],
                             cmap=label_colors[label])
    axes[row, 0].set_title(f'{label.upper()} — MFCC')
    axes[row, 0].set_ylabel('Coefficient')

    librosa.display.specshow(mel_d, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', y_axis='mel', ax=axes[row, 1],
                             cmap=label_colors[label])
    axes[row, 1].set_title(f'{label.upper()} — Mel-Spectrogram')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/results/feature_comparison.png', dpi=150)
plt.show()

## Step 7: Train / Dev / Test Split

In [ ]:
df_all = pd.read_csv(f'{BASE_DIR}/data/features_index.csv')

train_df, temp   = train_test_split(df_all, test_size=0.3,
                                    stratify=df_all['label'], random_state=42)
dev_df,   test_df = train_test_split(temp,  test_size=0.5,
                                    stratify=temp['label'],   random_state=42)

train_df.to_csv(f'{BASE_DIR}/data/train_index.csv', index=False)
dev_df.to_csv(  f'{BASE_DIR}/data/dev_index.csv',   index=False)
test_df.to_csv( f'{BASE_DIR}/data/test_index.csv',  index=False)

cfg = {
    'sample_rate': SAMPLE_RATE, 'n_mfcc': N_MFCC, 'n_mels': N_MELS,
    'hop_length': HOP_LENGTH, 'win_length': WIN_LENGTH, 'n_fft': N_FFT,
    'clip_duration': CLIP_DUR, 'base_dir': BASE_DIR,
    'total': len(df_all), 'train': len(train_df),
    'dev': len(dev_df), 'test': len(test_df),
    'class_counts': df_all['label'].value_counts().to_dict()
}
with open(f'{BASE_DIR}/data/phase1_summary.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Phase 1 Complete!')
print(f'  Train : {len(train_df)}')
print(f'  Dev   : {len(dev_df)}')
print(f'  Test  : {len(test_df)}')
print(f'\nSaved to {BASE_DIR}/data/')